In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_selection import SelectKBest, f_regression

print("\n================ TASK 1: HOUSE PRICE PREDICTION ================")

data = pd.DataFrame({
"bedrooms": [3, 2, 4, 3, 5, 2, 4, 3, 4, 5, 3, 2, 4, 3, 5],
"sqft": [1500, 1200, 2000, 1600, 3000, 1100, 1800, 1400, 2200, 2800, 1550, 1250, 1950, 1650, 2900],
"bathrooms": [2, 1, 3, 2, 4, 1, 3, 2, 3, 4, 2, 1, 3, 2, 4],
"age": [10, 20, 5, 15, 2, 25, 3, 18, 8, 1, 12, 22, 6, 14, 4],
"neighborhood": ["A", "B", "A", "C", "B", "A", "C", "B", "A", "C", "B", "A", "C", "B", "A"],
"price": [300000, 200000, 450000, 320000, 600000, 190000, 420000, 310000, 480000, 620000, 310000, 210000, 440000, 330000, 590000]
})

print("\nDataset shape:", data.shape)
print("Missing values:", data.isnull().sum().sum())

X = data.drop("price", axis=1)
y = data["price"]

column_transformer = ColumnTransformer([
('neighborhood', OneHotEncoder(drop='first'), ['neighborhood'])
], remainder='passthrough')

X_encoded = column_transformer.fit_transform(X)
feature_names = column_transformer.get_feature_names_out()
X_encoded_df = pd.DataFrame(X_encoded, columns=feature_names)

selector = SelectKBest(score_func=f_regression, k=4)
X_selected = selector.fit_transform(X_encoded_df, y)
selected_indices = selector.get_support()
selected_features = X_encoded_df.columns[selected_indices].tolist()
print("\nMost important features:", selected_features)

X_final = X_encoded_df[selected_features]
X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.3, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("\nMSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

feature_importance = pd.DataFrame({
"Feature": selected_features,
"Coefficient": model.coef_
})
print("\nFeature coefficients:")
print(feature_importance)

new_house = pd.DataFrame({
"bedrooms": [3],
"sqft": [1800],
"bathrooms": [2],
"age": [10],
"neighborhood": ["B"]
})
new_house_encoded = column_transformer.transform(new_house)
new_house_df = pd.DataFrame(new_house_encoded, columns=feature_names)
new_house_selected = new_house_df[selected_features]
prediction = model.predict(new_house_selected)
print("\nPredicted price for 3 bedrooms, 1800 sqft, 2 bathrooms, age 10, neighborhood B: $", round(prediction[0], 2))


================ TASK 1: HOUSE PRICE PREDICTION ================

Dataset shape: (15, 6)
Missing values: 0

Most important features: ['remainder__bedrooms', 'remainder__sqft', 'remainder__bathrooms', 'remainder__age']

MSE: 228507042.59128618
RMSE: 15116.44940425119
R2 Score: 0.9903988637566686

Feature coefficients:
                Feature   Coefficient
0   remainder__bedrooms  49569.525818
1       remainder__sqft     65.010114
2  remainder__bathrooms  49569.525818
3        remainder__age    607.146194

Predicted price for 3 bedrooms, 1800 sqft, 2 bathrooms, age 10, neighborhood B: $ 332626.05


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords', quiet=True)

print("\n================ TASK 2: SPAM CLASSIFICATION ================")

emails = [
    "Win money now click here", "Meeting scheduled for tomorrow at 10am", "Congratulations you won a prize claim now",
    "Please review the attached document", "Claim your free reward now immediately", "Your invoice is attached",
    "You have won a lottery ticket", "Project update meeting notes", "Urgent you need to verify your account",
    "Lunch break schedule changed", "Exclusive offer just for you", "Team building event next Friday",
    "Free gift card waiting for you", "Quarterly report review", "Congratulations you are a winner",
    "Staff meeting at 3pm", "Limited time offer buy now", "Reminder dentist appointment",
    "You have been selected for a prize", "Weekly status report"
]

labels = [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0]

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    stop_words = set(stopwords.words('english'))
    words = text.split()
    words = [word for word in words if word not in stop_words]
    stemmer = PorterStemmer()
    words = [stemmer.stem(word) for word in words]
    return ' '.join(words)

print("\nPreprocessing emails...")
emails_processed = [preprocess_text(email) for email in emails]

vectorizer = TfidfVectorizer(max_features=100)
X = vectorizer.fit_transform(emails_processed)

X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.3, random_state=42, stratify=labels)

model = SVC(kernel='linear', random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

new_emails = [
    "Win free iphone now click this link",
    "Meeting at 2pm tomorrow",
    "Congratulations you have won a free vacation",
    "Please find the attached report for your review",
]
new_emails_processed = [preprocess_text(email) for email in new_emails]
new_emails_vectorized = vectorizer.transform(new_emails_processed)
predictions = model.predict(new_emails_vectorized)

print("\nNew email predictions:")
for email, pred in zip(new_emails, predictions):
    print(f"  '{email}' -> {'SPAM' if pred == 1 else 'NOT SPAM'}")


================ TASK 2: SPAM CLASSIFICATION ================

Preprocessing emails...

Accuracy: 0.8333333333333334
Precision: 1.0
Recall: 0.6666666666666666
F1 Score: 0.8

Confusion Matrix:
[[3 0]
 [1 2]]

New email predictions:
  'Win free iphone now click this link' -> SPAM
  'Meeting at 2pm tomorrow' -> NOT SPAM
  'Congratulations you have won a free vacation' -> SPAM
  'Please find the attached report for your review' -> NOT SPAM


In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("\n================ TASK 3: CUSTOMER CLASSIFICATION ================")

np.random.seed(42)
n_samples = 200

data = pd.DataFrame({
"spending": np.random.normal(3000, 1500, n_samples),
"age": np.random.normal(35, 10, n_samples),
"visits": np.random.poisson(8, n_samples),
"frequency": np.random.poisson(3, n_samples)
})

data["spending"] = data["spending"].clip(100, 10000)
data["age"] = data["age"].clip(18, 70)
data["visits"] = data["visits"].clip(1, 30)
data["frequency"] = data["frequency"].clip(1, 10)

data.loc[data["spending"] > 4000, "value"] = 1
data.loc[data["spending"] <= 4000, "value"] = 0
data.loc[(data["visits"] > 10) & (data["frequency"] > 4), "value"] = 1

print("\nDataset shape:", data.shape)
print("Class distribution:\n", data["value"].value_counts())

print("\nMissing values before cleaning:", data.isnull().sum().sum())
data = data.dropna()

Q1 = data[["spending", "age", "visits", "frequency"]].quantile(0.25)
Q3 = data[["spending", "age", "visits", "frequency"]].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

before_outliers = len(data)
for col in ["spending", "age", "visits", "frequency"]:
    data = data[(data[col] >= lower_bound[col]) & (data[col] <= upper_bound[col])]
print(f"Outliers removed: {before_outliers - len(data)} rows")

X = data.drop("value", axis=1)
y = data["value"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42, stratify=y)

svm = SVC(kernel="linear", random_state=42)
svm.fit(X_train, y_train)
svm_pred = svm.predict(X_test)

print("\n=== SVM CLASSIFIER ===")
print("Accuracy:", accuracy_score(y_test, svm_pred))
print("Precision:", precision_score(y_test, svm_pred))
print("Recall:", recall_score(y_test, svm_pred))
print("F1 Score:", f1_score(y_test, svm_pred))
print("Coefficients (hyperplane normal vector):", svm.coef_[0])
print("Intercept:", svm.intercept_[0])

dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

print("\n=== DECISION TREE CLASSIFIER ===")
print("Accuracy:", accuracy_score(y_test, dt_pred))
print("Precision:", precision_score(y_test, dt_pred))
print("Recall:", recall_score(y_test, dt_pred))
print("F1 Score:", f1_score(y_test, dt_pred))

print("\nClassification rules extracted from decision tree:")
feature_names = ["spending", "age", "visits", "frequency"]
tree_rules = export_text(dt, feature_names=feature_names)
print(tree_rules)

new_customers = pd.DataFrame([
[5000, 30, 12, 5],
[1500, 45, 3, 1],
[3500, 25, 8, 3]
], columns=feature_names)

new_customers_scaled = scaler.transform(new_customers)
predictions = svm.predict(new_customers_scaled)

print("\n=== NEW CUSTOMER PREDICTIONS ===")
for i, (pred, row) in enumerate(zip(predictions, new_customers.values)):
    print(f"Customer {i+1}: Spending={row[0]}, Age={row[1]}, Visits={row[2]}, Frequency={row[3]} -> {'HIGH-VALUE' if pred == 1 else 'LOW-VALUE'}")


================ TASK 3: CUSTOMER CLASSIFICATION ================

Dataset shape: (200, 5)
Class distribution:
 value
0.0    152
1.0     48
Name: count, dtype: int64

Missing values before cleaning: 0
Outliers removed: 3 rows

=== SVM CLASSIFIER ===
Accuracy: 0.9666666666666667
Precision: 1.0
Recall: 0.8571428571428571
F1 Score: 0.9230769230769231
Coefficients (hyperplane normal vector): [2.32602852 0.12161731 0.07059309 0.28072094]
Intercept: -1.9333584048449057

=== DECISION TREE CLASSIFIER ===
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0

Classification rules extracted from decision tree:
|--- spending <= 0.83
|   |--- frequency <= 1.00
|   |   |--- class: 0.0
|   |--- frequency >  1.00
|   |   |--- visits <= 0.99
|   |   |   |--- class: 0.0
|   |   |--- visits >  0.99
|   |   |   |--- class: 1.0
|--- spending >  0.83
|   |--- class: 1.0


=== NEW CUSTOMER PREDICTIONS ===
Customer 1: Spending=5000, Age=30, Visits=12, Frequency=5 -> HIGH-VALUE
Customer 2: Spending=1500, Ag